# Getting Started with Snowflake Notebooks: Build an EDA and ML Pipeline

<div style="background: linear-gradient(135deg, #29B5E8, #1A73E8); padding: 40px 50px; border-radius: 12px; color: white;">
  <p style="margin: 8px 0 0 0; opacity: 0.9;">
    An end-to-end classification workflow: EDA → Modeling → Post-ML Analysis
  </p>
</div>

This notebook walks through a complete classification pipeline using the **Wine dataset** from scikit-learn — 178 samples, 13 chemical features, and 3 cultivar classes.

Step | Section | Highlights
--- | --- | ---
1 | **Setup** | Load Wine dataset, write to Snowflake temp table
2 | **EDA with SQL** | Class balance, per-class aggregations, ranked queries
3 | **EDA with Python** | Grouped box plots, 13×13 correlation heatmap
4 | **Machine Learning Modeling** | Train/test split, PCA scores plot, Random Forest, cross-validation
5 | **Post-ML Analysis** | Confusion matrix, feature importance, ROC curves, learning curve

**Runtime:** Container Runtime 2.6 (CPU) &nbsp;|&nbsp; **Estimated time:** ~2 minutes

## Setup

Load the **Wine dataset** from scikit-learn, connect to Snowflake, and prepare the data for SQL querying via Jinja templating.

- Load Wine dataset into `df` (178 samples, 13 features, 3 classes); sanitise column names into `df_snow`
- Install `ipywidgets` for interactive hyperparameter sliders
- Connect to Snowflake via `get_active_session()`; smoke-test with `CURRENT_TIMESTAMP()` and `CURRENT_VERSION()`
- SQL cells reference `df_snow` directly via `{{df_snow}}` Jinja templating — no explicit table upload needed

In [ ]:
import re
import pandas as pd
from sklearn.datasets import load_wine

# Load Wine dataset into a pandas DataFrame
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['cultivar'] = wine.target
df['cultivar_name'] = df['cultivar'].map({0: 'Cultivar 0', 1: 'Cultivar 1', 2: 'Cultivar 2'})

print(f"Dataset shape: {df.shape}")
print(f"Features: {list(wine.feature_names)}")
print(f"Classes: {list(wine.target_names)}")

# df_snow: column names sanitised for SQL (/ → _) so {{df_snow}} works in SQL cells
def safe_col(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name)

df_snow = df.rename(columns={c: safe_col(c) for c in df.columns})
print(f"\ndf_snow columns: {list(df_snow.columns)}")

In [ ]:
! pip install ipywidgets

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f'Connected as : {session.get_current_user()}')
print(f'Role         : {session.get_current_role()}')
print(f'Warehouse    : {session.get_current_warehouse()}')

# Smoke-test query
result = session.sql('SELECT CURRENT_TIMESTAMP() AS now, CURRENT_VERSION() AS sf_version').collect()
for row in result:
    print(f'Timestamp : {row["NOW"]}')
    print(f'SF version: {row["SF_VERSION"]}')

## EDA with SQL

Query the Wine dataset using Snowflake SQL. SQL cells reference `df_snow` directly via `{{df_snow}}` Jinja templating — no explicit table upload needed.

In Container Runtime 2.6, results captured with `%%sql -r <var>` are **Snowpark pandas (snowpandas) DataFrames** by default. Call `.to_pandas()` if a downstream operation requires a regular pandas DataFrame.

- **Class distribution** — count of samples per cultivar confirming balanced classes
- **Alcohol stats per cultivar** — min, avg, and max alcohol per cultivar revealing spread between classes
- **Top 9 by flavanoids** — highest-flavanoid samples ordered descending across all cultivars
- **Average feature values** — per-cultivar mean for alcohol, flavanoids, color intensity, and proline

In [ ]:
%%sql -r df_class_dist
-- Class distribution
SELECT
    cultivar,
    cultivar_name,
    COUNT(*) AS sample_count
FROM {{df_snow}}
GROUP BY cultivar, cultivar_name
ORDER BY cultivar

In [ ]:
%%sql -r df_alcohol_stats
-- Alcohol stats per cultivar
SELECT
    cultivar_name,
    ROUND(MIN(alcohol), 3) AS min_alcohol,
    ROUND(AVG(alcohol), 3) AS avg_alcohol,
    ROUND(MAX(alcohol), 3) AS max_alcohol
FROM {{df_snow}}
GROUP BY cultivar_name
ORDER BY cultivar_name

In [ ]:
%%sql -r df_top_flavanoids
-- Top 9 samples by flavanoid content
SELECT
    cultivar_name,
    ROUND(alcohol, 3)    AS alcohol,
    ROUND(flavanoids, 3) AS flavanoids,
    ROUND(proline, 0)    AS proline
FROM {{df_snow}}
ORDER BY flavanoids DESC
LIMIT 9

In [ ]:
%%sql -r df_feature_avgs
-- Average feature values per cultivar
SELECT
    cultivar_name,
    ROUND(AVG(alcohol), 3)          AS avg_alcohol,
    ROUND(AVG(flavanoids), 3)       AS avg_flavanoids,
    ROUND(AVG(color_intensity), 3)  AS avg_color_intensity,
    ROUND(AVG(proline), 3)          AS avg_proline
FROM {{df_snow}}
GROUP BY cultivar_name
ORDER BY cultivar_name

## EDA with Python

Python-based EDA focuses on the *shape* of the data — distributions across classes and feature-to-feature relationships.

- **Grouped box plots** — 4×4 grid of all 13 feature distributions broken out by cultivar
- **Correlation heatmap** — full 13×13 Pearson correlation matrix with annotated coefficients
- **Descriptive statistics** — `describe()` table of count, mean, std, min, quartiles, and max for every feature
- **Pairplot** — scatter matrix of the 5 most discriminative features coloured by cultivar class

In [ ]:
import matplotlib.pyplot as plt

# Grouped box plots: each feature broken out by cultivar
features = wine.feature_names
n_cols = 4
n_rows = (len(features) + n_cols - 1) // n_cols  # ceil division

palette = {0: '#29B5E8', 1: '#FF6B35', 2: '#4CAF50'}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    data_by_class = [df[df['cultivar'] == c][feat].values for c in [0, 1, 2]]
    bp = ax.boxplot(data_by_class, patch_artist=True, tick_labels=['C0', 'C1', 'C2'],
                    medianprops=dict(color='black', linewidth=1.5))
    colors = ['#29B5E8', '#FF6B35', '#4CAF50']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('Cultivar')
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# Hide unused subplots
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions by Cultivar (Grouped Box Plots)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns

# 13x13 correlation heatmap
numeric_df = df[list(wine.feature_names)]
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle + diagonal
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.4,
    annot_kws={'size': 7},
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=ax
)
ax.set_title('Feature Correlation Matrix (13 × 13)', fontsize=14, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Descriptive statistics — count, mean, std, min, quartiles, max for every feature
stats = df[list(wine.feature_names)].describe().T.round(3)
print(stats.to_string())

In [ ]:
# Pairplot for the 5 most discriminative features, coloured by cultivar
# These features are known to separate the three classes well
key_features = ['alcohol', 'flavanoids', 'color_intensity', 'proline',
                'od280/od315_of_diluted_wines']
pair_df = df[key_features + ['cultivar_name']].copy()

palette = {'Cultivar 0': '#29B5E8', 'Cultivar 1': '#FF6B35', 'Cultivar 2': '#4CAF50'}
g = sns.pairplot(
    pair_df,
    hue='cultivar_name',
    palette=palette,
    plot_kws={'alpha': 0.6, 's': 25},
    diag_kind='kde'
)
g.figure.suptitle('Pairplot — Key Features by Cultivar', y=1.02,
                  fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Machine Learning Modeling

Preprocess the data, visualise the train/test split in PCA space, train a Random Forest classifier, and evaluate with cross-validation.

- **Train/test split** — 80/20 stratified split; features scaled with `StandardScaler` fitted on train, applied to test
- **PCA scores plot** — 2-component PCA visualised side-by-side by train/test split and by cultivar class
- **Hyperparameter sliders** — `ipywidgets` sliders for `n_estimators` (10–500) and `max_depth` (1–20)
- **Train Random Forest** — fits `RandomForestClassifier` from slider values; reports test accuracy and 5-fold CV scores
- **Classification report** — per-class precision, recall, and F1-score on the test set

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df[list(wine.feature_names)].values
y = df['cultivar'].values

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set: {X_train_scaled.shape[0]} samples")
print(f"Test set:  {X_test_scaled.shape[0]} samples")

In [ ]:
from sklearn.decomposition import PCA

# Source the full dataset directly from df to avoid any variable aliasing issues
X_full = df[list(wine.feature_names)].values  # always 178 × 13
y_full = df['cultivar'].values
X_full_scaled = scaler.transform(X_full)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_full_scaled)
var_explained = pca.explained_variance_ratio_ * 100

# Build split labels using the same random_state as preprocessing
# dtype=object avoids numpy string truncation ('Train' 5-char vs 'Test' 4-char)
train_idx, _ = train_test_split(
    np.arange(len(X_full)), test_size=0.2, random_state=42, stratify=y_full
)
split_labels = np.full(len(X_full), 'Test', dtype=object)
split_labels[train_idx] = 'Train'
print(f"Train: {(split_labels == 'Train').sum()} | Test: {(split_labels == 'Test').sum()} samples")

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'cultivar': y_full,
    'split': split_labels
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Train/Test split — draw Test first (behind), Train second (on top)
for split, color, size, alpha, zorder in [
    ('Test',  '#FF6B35', 70, 0.85, 2),
    ('Train', '#29B5E8', 50, 0.70, 3),
]:
    mask = pca_df['split'] == split
    axes[0].scatter(
        pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
        color=color, label=f'{split} (n={mask.sum()})',
        s=size, alpha=alpha, edgecolors='white', linewidths=0.5, zorder=zorder
    )
axes[0].set_title('PCA Scores — Train / Test Split', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_explained[0]:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]:.1f}% var)')
axes[0].legend(title='Split', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: Cultivar class
class_colors = {0: '#29B5E8', 1: '#FF6B35', 2: '#4CAF50'}
for cls, color in class_colors.items():
    mask = pca_df['cultivar'] == cls
    axes[1].scatter(
        pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
        color=color, label=f'Cultivar {cls} (n={mask.sum()})',
        s=50, alpha=0.75, edgecolors='white', linewidths=0.5
    )
axes[1].set_title('PCA Scores — Cultivar Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var_explained[0]:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]:.1f}% var)')
axes[1].legend(title='Cultivar', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

n_estimators_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='n_estimators:',
    style={'description_width': 'initial'},
    continuous_update=False
)

max_depth_slider = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description='max_depth:',
    style={'description_width': 'initial'},
    continuous_update=False
)

print('Adjust sliders then run the next cell to train the model.')
display(n_estimators_slider, max_depth_slider)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Read hyperparameters from widgets
n_estimators = n_estimators_slider.value
max_depth = max_depth_slider.value
print(f'Training RandomForest with n_estimators={n_estimators}, max_depth={max_depth}')

# Train Random Forest
rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
rf.fit(X_train_scaled, y_train)

# Test set accuracy
test_accuracy = rf.score(X_test_scaled, y_test)
print(f"Test set accuracy: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")

# 5-fold cross-validation on full dataset
cv_scores = cross_val_score(rf, scaler.transform(X), y, cv=5, scoring='accuracy')
print(f"\n5-Fold Cross-Validation:")
print(f"  Scores: {[f'{s:.3f}' for s in cv_scores]}")
print(f"  Mean:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
from sklearn.metrics import classification_report

y_pred = rf.predict(X_test_scaled)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## Post-ML Analysis

Diagnostics and interpretation: where the model makes mistakes, which features drive predictions, how it separates classes, and whether it needs more data.

- **Confusion matrix** — seaborn heatmap of true vs predicted labels on the test set
- **Feature importances** — horizontal bar chart of mean decrease in impurity sorted ascending
- **ROC curves** — side-by-side one-vs-rest AUC plots for the test set and 5-fold CV out-of-fold predictions, with shaded area under each curve
- **Learning curve** — training vs CV accuracy with ±1 std shading as training set size increases

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    cmap='Blues',
    xticklabels=wine.target_names,
    yticklabels=wine.target_names,
    linewidths=0.5,
    cbar_kws={'label': 'Count'},
    ax=ax
)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance horizontal bar chart
importances = rf.feature_importances_
feature_names = wine.feature_names
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(range(len(sorted_idx)), importances[sorted_idx], align='center',
               color='#29B5E8', alpha=0.8, edgecolor='white')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=10)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_title('Random Forest Feature Importances', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# Annotate values
for i, (bar, val) in enumerate(zip(bars, importances[sorted_idx])):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_predict

# Binarize labels for one-vs-rest ROC
y_bin = label_binarize(y, classes=[0, 1, 2])
X_scaled_all = scaler.transform(X)
_, X_test_all, _, y_test_bin = train_test_split(
    X_scaled_all, y_bin, test_size=0.2, random_state=42, stratify=y
)
y_score = rf.predict_proba(X_test_scaled)

# Out-of-fold CV probabilities for all samples
y_cv_score = cross_val_predict(
    RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42),
    X_scaled_all, y, cv=5, method='predict_proba'
)

colors = ['#29B5E8', '#FF6B35', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

panels = [
    (y_score,    y_test_bin, 'ROC Curves — Test Set'),
    (y_cv_score, y_bin,      'ROC Curves — 5-Fold CV (Out-of-Fold)'),
]

for ax, (scores, labels, title) in zip(axes, panels):
    for i, (cls_name, color) in enumerate(zip(wine.target_names, colors)):
        fpr, tpr, _ = roc_curve(labels[:, i], scores[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{cls_name} (AUC = {roc_auc:.3f})')
        ax.fill_between(fpr, tpr, alpha=0.1, color=color)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42),
    scaler.transform(X), y,
    cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='#29B5E8', linewidth=2, label='Training accuracy')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='#29B5E8')
ax.plot(train_sizes, val_mean, 's-', color='#FF6B35', linewidth=2, label='CV accuracy')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='#FF6B35')
ax.set_xlabel('Training Set Size', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title(f'Learning Curve — Random Forest (n_estimators={n_estimators}, max_depth={max_depth})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim([0.7, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated a complete end-to-end classification pipeline on the Wine dataset:

- **Setup** — loaded the Wine dataset into a pandas DataFrame, connected to Snowflake via `get_active_session()`, and sanitised column names for SQL compatibility; SQL cells reference `df_snow` directly via `{{df_snow}}` Jinja templating — no explicit table upload needed
- **SQL EDA** — queried `df_snow` via Jinja templating to verify class balance, compare alcohol stats across cultivars, surface top samples by flavanoid content, and compare per-cultivar feature averages; results captured with `%%sql -r` are Snowpark pandas (snowpandas) DataFrames — call `.to_pandas()` for downstream pandas operations
- **Python EDA** — grouped box plots revealed feature separability by cultivar; the 13×13 correlation heatmap highlighted collinear features (e.g. flavanoids ↔ total_phenols); a pairplot of the 5 most discriminative features confirmed near-linear class separation
- **PCA scores plot** — confirmed the 80/20 stratified train/test split is representative and that cultivars are largely linearly separable in 2D PCA space
- **Random Forest** — trained with interactive `ipywidgets` sliders for `n_estimators` and `max_depth`; achieved high test accuracy; 5-fold cross-validation confirmed the result generalises beyond the single split
- **Confusion matrix** — identified which cultivars (if any) are confused with each other on the test set
- **Feature importances** — ranked chemical features by mean decrease in impurity; proline, color_intensity, and flavanoids are typically the most discriminative
- **ROC curves** — side-by-side AUC plots for the test set and 5-fold CV out-of-fold predictions with shaded areas confirmed strong one-vs-rest discrimination and consistent generalisation
- **Learning curve** — small gap between training and CV accuracy indicates the model is not overfitting and is unlikely to benefit significantly from more data

### Next Steps
- Compare with other classifiers: `SVC`, `GradientBoostingClassifier`, `LogisticRegression`
- Tune hyperparameters with `GridSearchCV` or `RandomizedSearchCV`
- Register the trained model in the **Snowflake ML Model Registry** for versioning and deployment
- Score new wine samples directly in Snowflake SQL using a deployed model endpoint

## Natural Language Prompts

Use these prompts to reproduce or extend each section of this notebook with an AI coding assistant.

---

### Setup

> Load the scikit-learn Wine dataset into a pandas DataFrame. Sanitise column names by replacing / with _ so they are safe to use in SQL. Print the dataset shape, feature names, and class names.

---

### EDA with SQL

> Using a Snowpark session in a Snowflake Notebook, write four SQL cells that reference a pandas DataFrame via Jinja templating ({{df_snow}}): (1) count samples per cultivar with percentage of total using a window function, (2) compute a five-number summary (min, Q1, median, Q3, max) of alcohol content grouped by cultivar using PERCENTILE_CONT, (3) rank the top 3 samples per cultivar by flavanoid content using RANK() OVER (PARTITION BY), and (4) compute per-feature average by cultivar using a single-scan UNPIVOT + PIVOT instead of multiple UNION ALL subqueries.

---

### EDA with Python

> Using matplotlib and seaborn, produce three visualisations for the Wine dataset: (1) a grid of grouped box plots showing the distribution of every feature broken out by cultivar, (2) a lower-triangle 13x13 Pearson correlation heatmap with annotated coefficients, and (3) a pairplot of the five most discriminative features coloured by cultivar class.

---

### Machine Learning Modeling

> Split the Wine dataset 80/20 with stratification and scale features using StandardScaler. Fit a PCA with 2 components and plot the scores coloured by (a) train/test split and (b) cultivar class in side-by-side scatter plots. Add ipywidgets IntSlider widgets for n_estimators (range 10-500, step 10) and max_depth (range 1-20), then train a RandomForestClassifier reading those slider values, report test-set accuracy, and run 5-fold cross-validation on the full dataset.

---

### Post-ML Analysis

> After training a Random Forest on the Wine dataset, produce four evaluation plots: (1) a seaborn heatmap confusion matrix for the test set, (2) a horizontal bar chart of feature importances sorted ascending, (3) one-vs-rest ROC curves with AUC scores for all three cultivar classes on a single axes, and (4) a learning curve showing mean training and cross-validation accuracy with +/-1 std shading as training set size increases. The learning curve title should reflect the current n_estimators and max_depth values from the ipywidgets sliders.